# 01 — UCI EEG Dataset Exploration (Alcoholism Study)

**Purpose:** Explore the UCI EEG alcohol dataset (122 subjects: 77 alcoholic, 45 control).
Visualize raw signals, compare groups, check data quality.

**Dataset:** UCI ML Repository — 64 channels, 256 Hz, visual stimulus paradigm.

In [ ]:
import sys
sys.path.insert(0, '../..')

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from classifiers.addiction.data import load_uci_dataset, AddictionConfig, dataset_to_epochs

plt.style.use('dark_background')
UCI_DIR = Path('../../data/raw/uci_eeg')

## 1. Load Dataset and Overview

In [ ]:
config = AddictionConfig(data_dir=UCI_DIR)
dataset = load_uci_dataset(config)

n_subjects = len(np.unique(dataset.subjects))
n_alcoholic = (dataset.labels == 1).sum()
n_control = (dataset.labels == 0).sum()

print(f'Total subjects: {n_subjects}')
print(f'Total trials: {len(dataset.labels)}')
print(f'Alcoholic trials: {n_alcoholic} ({n_alcoholic/len(dataset.labels)*100:.1f}%)')
print(f'Control trials: {n_control} ({n_control/len(dataset.labels)*100:.1f}%)')
print(f'Data shape: {dataset.data.shape} (trials × channels × samples)')

## 2. Raw Signal Comparison: Alcoholic vs Control

In [ ]:
# Pick one alcoholic and one control trial
alc_idx = np.where(dataset.labels == 1)[0][0]
ctrl_idx = np.where(dataset.labels == 0)[0][0]

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
times = np.arange(256) / 256.0

# Plot first 8 channels
for ch in range(8):
    axes[0].plot(times, dataset.data[alc_idx, ch] + ch * 10, linewidth=0.5, color='#E57373')
    axes[1].plot(times, dataset.data[ctrl_idx, ch] + ch * 10, linewidth=0.5, color='#4FC3F7')

axes[0].set_title('Alcoholic Subject — Single Trial', color='#E57373')
axes[1].set_title('Control Subject — Single Trial', color='#4FC3F7')
axes[1].set_xlabel('Time (s)')
for ax in axes:
    ax.set_ylabel('Channel')
plt.tight_layout()
plt.show()

## 3. Group Statistics

In [ ]:
# Trials per subject
unique_subjects = np.unique(dataset.subjects)
trials_per_subject = [np.sum(dataset.subjects == s) for s in unique_subjects]
labels_per_subject = [dataset.labels[dataset.subjects == s][0] for s in unique_subjects]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Group counts
n_alc_subj = sum(1 for l in labels_per_subject if l == 1)
n_ctrl_subj = sum(1 for l in labels_per_subject if l == 0)
bars = ax1.bar(['Alcoholic', 'Control'], [n_alc_subj, n_ctrl_subj],
               color=['#E57373', '#4FC3F7'], edgecolor='#333', linewidth=2)
ax1.set_ylabel('Number of Subjects')
ax1.set_title('Subject Distribution')
for bar, val in zip(bars, [n_alc_subj, n_ctrl_subj]):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            str(val), ha='center')

# Trials per subject histogram
ax2.hist(trials_per_subject, bins=20, color='#888', edgecolor='#333', linewidth=1)
ax2.set_xlabel('Trials per Subject')
ax2.set_ylabel('Count')
ax2.set_title('Trials per Subject Distribution')
plt.tight_layout()
plt.show()